# Clase 12 - Ingenieria de datos: formatos de archivo y proceso ETL

**Objetivos de la clase**

- Entender que es el proceso **ETL** (Extraer, Transformar, Cargar) y por que es el flujo mental detras de casi cualquier trabajo con datos.
- Leer y escribir archivos en los formatos mas comunes: **CSV**, **JSON**, **XML** y (opcionalmente) **XLSX**.
- Aplicar una primera exploracion de un DataFrame con `.info()`, `.describe()`, `.isnull()` y `.dtypes`.

Todo el codigo de este notebook es autocontenido: no depende de descargar ningun dataset externo. Vamos a crear nuestros propios datos ficticios (tipo "dataset medico") con NumPy y Pandas, usando una semilla fija para que el resultado sea siempre el mismo.

## 1. El proceso ETL

Cada vez que trabajas con datos, en el fondo estas haciendo tres cosas:

1. **Extraer (Extract)**: conseguir los datos desde donde esten (un archivo CSV, un JSON de una API, un Excel que te mandaron por correo, etc.). En este paso todavia no tocas el contenido, solo lo traes a tu entorno de trabajo.
2. **Transformar (Transform)**: limpiar y unificar. Eliminar lo que no sirve, convertir tipos de dato, unificar formatos entre distintas fuentes.
3. **Cargar (Load)**: dejar los datos ya transformados listos para analizar -en tu caso, casi siempre un DataFrame de Pandas.

Cada vez que haces `df = pd.read_csv(...)` seguido de `df.dropna()` o de renombrar columnas, ya estas haciendo ETL a pequena escala. El resto de esta clase es, en el fondo, un catalogo de tecnicas para el paso de "Extraer" segun el formato en que vengan los datos.

## 2. Formato CSV

El **CSV** (comma-separated values) es texto plano tabular: cada linea es un registro, y los campos van separados por comas. Ya conoces `pd.read_csv()` y `df.to_csv()`; en esta seccion repasamos dos detalles utiles en la practica y trabajamos con un dataset propio.

Primero, creamos un DataFrame ficticio tipo "dataset clinico" con NumPy, usando una semilla fija (`np.random.seed`) para que los resultados sean reproducibles.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 15

pacientes = pd.DataFrame({
    "id_paciente": range(1, n + 1),
    "embarazos": np.random.randint(0, 6, size=n),
    "glucosa": np.random.randint(80, 200, size=n).astype(float),
    "presion_arterial": np.random.randint(60, 100, size=n),
    "imc": np.round(np.random.uniform(18.5, 40.0, size=n), 1),
    "edad": np.random.randint(21, 65, size=n),
})

# Insertamos a proposito un par de valores faltantes para practicar .isnull() mas adelante
pacientes.loc[3, "glucosa"] = np.nan
pacientes.loc[7, "imc"] = np.nan

pacientes.head()

### Cuando el CSV no trae encabezados

Si un CSV no incluye una fila de encabezado, `read_csv` toma por defecto la primera fila de datos como si fueran nombres de columna -un error clasico. Se soluciona con `header=None` y asignando tu mismo los nombres despues, usando el atributo `.columns`.

In [ ]:
# Simulamos un CSV "sin encabezados" escribiendolo nosotros mismos
direcciones_sin_encabezado = pd.DataFrame([
    ["Rita", "Perez", "Av. Libertad 100", "Cordoba", "Cordoba", "5000"],
    ["Juan", "Gomez", "Calle Falsa 123", "Rosario", "Santa Fe", "2000"],
])
direcciones_sin_encabezado.to_csv("direcciones.csv", index=False, header=False)

# Al leerlo con header=None, Pandas no asume que la primera fila son nombres de columna
df_direcciones = pd.read_csv("direcciones.csv", header=None)
df_direcciones.columns = ["Nombre", "Apellido", "Calle", "Ciudad", "Provincia", "Codigo_Postal"]
df_direcciones

### La funcion `.transform()`

`.transform()` aplica una funcion a cada elemento de un DataFrame y devuelve un DataFrame nuevo con el resultado, sin necesidad de escribir un bucle. Es tipico del paso de "Transformar" del ETL cuando necesitas aplicar la misma operacion a toda una tabla numerica de una vez.

In [ ]:
df_ejemplo = pd.DataFrame(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), columns=["a", "b", "c"])

# Sumar 10 a cada elemento
df_mas_10 = df_ejemplo.transform(func=lambda x: x + 10)
print(df_mas_10)

# Aplicar una funcion por nombre en vez de una lambda
df_sqrt = df_ejemplo.transform(func=["sqrt"])
print(df_sqrt)

### Guardar y volver a leer el CSV (el paso de "Cargar")

`index=False` evita que Pandas agregue una columna extra con el numero de fila -sin este argumento, cada vez que guardas y vuelves a leer el archivo acumulas una columna de indice de mas.

In [ ]:
pacientes.to_csv("pacientes.csv", index=False)

pacientes_desde_csv = pd.read_csv("pacientes.csv")
pacientes_desde_csv.head()

## 3. Formato JSON

**JSON** guarda pares clave-valor y listas, igual que un diccionario de Python. Convertir un objeto de Python a JSON se llama **serializar**; el proceso inverso se llama **deserializar**. Python trae todo esto en el paquete integrado `json`, sin instalar nada.

- `json.dump(objeto, archivo)` serializa directo a un archivo abierto.
- `json.dumps(objeto)` serializa a un string en memoria (util para imprimir o enviar por otro canal).
- `json.load(archivo)` deserializa desde un archivo abierto.
- `json.loads(string)` deserializa desde un string en memoria.

Memoriza el par: `dump`/`dumps` van de Python hacia JSON, `load`/`loads` van de JSON hacia Python.

In [ ]:
import json

paciente_dict = {
    "id_paciente": 1,
    "nombre": "Ana",
    "edad": 34,
    "mediciones": {"glucosa": 105, "imc": 24.3},
}

# Serializar directo a archivo con dump()
with open("paciente.json", "w") as f:
    json.dump(paciente_dict, f, indent=4)

# Serializar a string con dumps() -util para imprimir con formato legible
paciente_json_str = json.dumps(paciente_dict, indent=4)
print(paciente_json_str)

In [ ]:
# Deserializar desde archivo con load()
with open("paciente.json", "r") as f:
    paciente_cargado = json.load(f)

print(paciente_cargado)
print(type(paciente_cargado))

# Deserializar desde string con loads()
paciente_desde_string = json.loads(paciente_json_str)
print(paciente_desde_string == paciente_cargado)

Pandas tambien tiene sus propias funciones `pd.read_json()` / `df.to_json()`, que siguen el mismo patron que `read_csv`/`to_csv` pero para JSON.

In [ ]:
pacientes.to_json("pacientes.json", orient="records", indent=2)

pacientes_desde_json = pd.read_json("pacientes.json")
pacientes_desde_json.head()

## 4. Formato XLSX (opcional)

**XLSX** es el formato de Excel. El flujo es identico al de CSV, solo que con `pd.read_excel()` / `df.to_excel()`. Para que Pandas pueda leer y escribir XLSX necesita la libreria `openpyxl` instalada por debajo.

> **Nota:** esta seccion detecta si `openpyxl` esta disponible en el entorno. Si no lo esta, el codigo no falla: simplemente imprime un aviso y muestra cual seria el patron de codigo a usar.

In [ ]:
try:
    import openpyxl  # noqa: F401

    pacientes.to_excel("pacientes.xlsx", index=False)
    pacientes_desde_excel = pd.read_excel("pacientes.xlsx")
    print("openpyxl esta disponible: se escribio y se leyo pacientes.xlsx correctamente.")
    print(pacientes_desde_excel.head())
except ImportError:
    print("openpyxl no esta instalado en este entorno.")
    print("Para trabajar con XLSX instala la libreria con: pip install openpyxl")
    print("El patron de codigo, una vez instalado, es:")
    print('  pacientes.to_excel("pacientes.xlsx", index=False)')
    print('  pd.read_excel("pacientes.xlsx")')

## 5. Formato XML

Pandas no trae de fabrica una forma completa de leer y escribir XML. Para eso usamos el modulo integrado `xml.etree.ElementTree`, que representa el documento XML como un arbol de nodos que se puede recorrer.

### Escribir un XML desde cero

In [ ]:
import xml.etree.ElementTree as ET

pacientes_xml = ET.Element("pacientes")
for _, fila in pacientes.head(3).iterrows():
    paciente_el = ET.SubElement(pacientes_xml, "paciente")
    ET.SubElement(paciente_el, "id").text = str(fila["id_paciente"])
    ET.SubElement(paciente_el, "edad").text = str(fila["edad"])
    ET.SubElement(paciente_el, "imc").text = str(fila["imc"])

arbol = ET.ElementTree(pacientes_xml)
arbol.write("pacientes.xml")
print("Archivo pacientes.xml creado")

### Leer un XML manualmente con ElementTree

El patron es: parsear el archivo, definir las columnas esperadas, y recorrer cada nodo extrayendo el texto de cada subelemento.

In [ ]:
import xml.etree.ElementTree as etree

tree = etree.parse("pacientes.xml")
root = tree.getroot()
columnas = ["id", "edad", "imc"]

filas = []
for nodo in root:
    filas.append({
        "id": nodo.find("id").text,
        "edad": nodo.find("edad").text,
        "imc": nodo.find("imc").text,
    })

pacientes_desde_xml = pd.DataFrame(filas, columns=columnas)
pacientes_desde_xml

### Leer un XML con `pd.read_xml()`

Si la estructura del XML es simple y repetitiva, `pd.read_xml()` ahorra el recorrido manual: le pasas un `xpath` indicando que nodos pasan a ser filas.

Por defecto, `pd.read_xml()` usa el parser `lxml` (una libreria externa que hay que instalar aparte). Como en este notebook queremos depender solo de la libreria estandar, pasamos `parser="etree"` para que use `xml.etree.ElementTree` por debajo -en ese caso el `xpath` se escribe en forma relativa (`"./paciente"`).

In [ ]:
pacientes_desde_xml_pd = pd.read_xml("pacientes.xml", xpath="./paciente", parser="etree")
pacientes_desde_xml_pd

## 6. Primera exploracion de un dataset

Con el ETL y los formatos cubiertos, cerramos aplicando las herramientas de exploracion de Pandas sobre nuestro DataFrame `pacientes`. Esta fase es, en el fondo, la validacion de que el ETL salio bien: si acabas de cargar un dataset, lo primero que haces es comprobar que la carga trajo lo que esperabas.

### `.info()`: resumen estructural

In [ ]:
pacientes.info()

`.info()` muestra cuantas filas tiene el DataFrame, el nombre y tipo de cada columna, y cuantos valores no nulos hay en cada una. Es tu primer chequeo de salud: si una columna que esperabas numerica aparece como `object`, o el conteo de no-nulos es menor que el total de filas, ya sabes donde mirar.

### `.describe()`: estadistica descriptiva

In [ ]:
pacientes.describe()

`.describe()` da estadistica descriptiva de las columnas numericas: cuenta, media, desviacion estandar, minimo, maximo y los percentiles 25/50/75.

### `.isnull()`: deteccion de valores faltantes

In [ ]:
missing_data = pacientes.isnull()
missing_data.head()

In [ ]:
# Contar cuantos True (valores faltantes) hay en cada columna
for column in missing_data.columns.values.tolist():
    print(column)
    print(missing_data[column].value_counts())
    print("")

### `.dtypes`: tipo de dato de cada columna

In [ ]:
pacientes.dtypes

Si todo esto sale como esperabas -tipos correctos, valores faltantes identificados y explicados-, ya tienes la confirmacion de que el paso de "Extraer y Transformar" del ETL termino bien y estas listo para pasar al analisis.

## Resumen de la clase

- **ETL** es Extraer, Transformar y Cargar: el flujo mental detras de casi todo trabajo con datos.
- **CSV**: texto plano tabular. `pd.read_csv()` (con `header=None` si no trae encabezados) / `df.to_csv(..., index=False)`.
- **JSON**: pares clave-valor y listas, igual que un diccionario de Python. Serializar con `json.dump()`/`json.dumps()`, deserializar con `json.load()`/`json.loads()`. Tambien existen `pd.read_json()`/`df.to_json()`.
- **XLSX**: hojas de calculo de Excel. `pd.read_excel()`/`df.to_excel()`, requieren `openpyxl` instalado.
- **XML**: arbol de nodos. Se recorre a mano con `xml.etree.ElementTree`, o se usa `pd.read_xml(archivo, xpath=...)` si la estructura es simple.
- `.info()`, `.describe()`, `.isnull()` y `.dtypes` son tu primer chequeo de salud de cualquier dataset recien cargado.